# 🚀 BIST-Finans-LLM: Google Colab Pro ile Açık Kaynak Finans Modeli Eğitimi (QLoRA / Unsloth)

Bu notebook, **Borsa İstanbul (BIST 100)** şirketlerinin resmi faaliyet raporları, bilanço dipnotları, DuPont kârlılığı, Altman Z-skorları ve KAP bildirimleriyle açık kaynaklı bir modeli (**Qwen 2.5 7B-Instruct** veya **Llama 3.1 8B-Instruct**) kurumsal bir **BIST Kıdemli Finansal Analist Asistanına** dönüştürmek için hazırlanmıştır.

### 🌟 Neden Google Colab Pro?
- **Yüksek Hız:** Colab Pro'daki A100 veya L4 GPU ile eğitim **8 - 12 dakika** içinde tamamlanır.
- **Hafıza Optimizasyonu (Unsloth):** %70 daha az VRAM harcar, 4-bit QLoRA sayesinde standart 16GB GPU'lara dahi rahatça sığar.
- **Canlı API Köprüsü (ngrok):** Eğitim biter bitmez aynı notebook içinde yerel bir FastAPI sunucusu açılır ve canlı public URL ile bilgisayarınızdaki Streamlit Chatbot projenize bağlanır!

### 1. Adım: GPU Donanım ve Ortam Kontrolü

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Kullanılabilir mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. Adım: Unsloth ve Gerekli Kütüphanelerin Kurulumu

In [ ]:
# Unsloth, bitsandbytes ve gerekli kutuphanelerin kurulumu
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth_zoo bitsandbytes
!pip install pyngrok fastapi uvicorn nest_asyncio datasets trl peft accelerate


### 3. Adım: Taban Model ve Tokenizer Yükleme (4-bit QLoRA)
Finansal muhakeme ve Türkçe uyumu çok yüksek olan `Qwen/Qwen2.5-7B-Instruct` modelini 4-bit kuantize olarak yüklüyoruz.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None: GPU'ya göre Float16 veya Bfloat16'yı otomatik seçer
load_in_4bit = True  # 4-bit QLoRA ile VRAM tasarrufu

# Model Adı: "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" veya "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

print(f"Model yükleniyor: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Taban model başarıyla GPU belleğine alındı!")

### 4. Adım: LoRA / PEFT Parametrelerinin Yapılandırılması
Tüm ağırlıkları değil, sadece dikkat (attention) ve MLP katmanlarındaki adapter matrislerini eğitiyoruz (parametrelerin yalnızca %1-2'si güncellenir).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA Rank (16 veya 32 idealdir)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,  # 0 optimize edilmiştir
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✅ LoRA adapter katmanları bağlandı!")

### 5. Adım: BIST 100 Eğitim Veri Kümesini Yükleme (`finetune_bist_dataset.jsonl`)
Projemizin ürettiği `data/finetune_bist_dataset.jsonl` dosyasını sol panelden Colab'a yükleyin veya Google Drive'dan bağlayın.

In [ ]:
from datasets import load_dataset
import os

dataset_filename = "finetune_bist_dataset.jsonl"
if not os.path.exists(dataset_filename):
    print(f"⚠️ '{dataset_filename}' bulunamadı. Lütfen sol dosya panelinden bu dosyayı Colab'a yükleyin.")
else:
    dataset = load_dataset("json", data_files=dataset_filename, split="train")
    print(f"✅ Eğitim veri kümesi yüklendi: Toplam {len(dataset)} BIST finansal talimat örneği!")

    # Qwen ChatML şablonuna çevir
    def format_chatml(examples):
        instructions = examples["instruction"]
        inputs = examples["input"]
        outputs = examples["output"]
        texts = []
        for inst, inp, out in zip(instructions, inputs, outputs):
            user_content = f"{inst}\n\n{inp}" if inp else inst
            text = f"<|im_start|>system\nSen Borsa İstanbul (BIST) uzmanı, finansal raporları, bilançoları ve rasyoları analiz eden kıdemli bir analistsin.<|im_end|>\n<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
            texts.append(text)
        return {"text": texts}

    formatted_dataset = dataset.map(format_chatml, batched=True)
    print("Örnek girdi:", formatted_dataset[0]["text"][:300], "...")

### 6. Adım: Model Eğitimi (SFTTrainer)
Colab Pro GPU ile eğitim başlatılıyor...

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,  # Colab Pro A100'de yaklaşık 6-8 dakika
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="bist_finans_lora_outputs",
    ),
)

print("🚀 Eğitim Başlatılıyor...")
trainer_stats = trainer.train()
print("🎉 Model Başarıyla Eğitildi!")

### 7. Adım: Eğitilmiş BIST Finans Modelini Test Etme

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "<|im_start|>system\nSen Borsa İstanbul (BIST) uzmanı, finansal raporları, bilançoları ve rasyoları analiz eden kıdemli bir analistsin.<|im_end|>\n<|im_start|>user\nTHYAO hissesinin DuPont kârlılık sürücüleri ve finansal sağlamlık Z-skoru nedir?<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.3,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.18,
    no_repeat_ngram_size=4,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True,
)
response_text = tokenizer.batch_decode(outputs)[0]

print("\n--- MODELİN YANITI ---\n")
print(response_text.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", ""))


### 8. Adım: Modeli Google Drive'a Kaydetme (AI Klasörü)
Eğitilen LoRA ağırlıklarını Google Drive'ınızdaki **'AI'** klasörüne kalıcı olarak kaydedebilirsiniz.
Böylece Colab oturumu kapansa bile modeliniz Google Drive'ınızda güvende kalır!

In [ ]:
# 1. Google Drive'ı bağla (Mount Drive)
from google.colab import drive
import os

drive.mount('/content/drive')

# 2. Google Drive üzerinde 'AI' klasörü ve model kayıt dizini oluştur
ai_folder = '/content/drive/MyDrive/AI'
lora_save_path = os.path.join(ai_folder, 'bist_finans_lora_weights')
os.makedirs(lora_save_path, exist_ok=True)

# 3. LoRA adapter ağırlıklarını ve tokenizer'ı doğrudan Google Drive'a kaydet
print(f"💾 Model Google Drive'a kaydediliyor: {lora_save_path} ...")
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)
print(f"🎉 TEBRİKLER! LoRA ağırlıkları başarıyla Google Drive'ınıza kaydedildi: {lora_save_path}")

# İsteğe bağlı: Ollama için GGUF formatına dönüştürüp Google Drive'a kaydetme (Q4_K_M)
# gguf_save_path = os.path.join(ai_folder, 'bist_finans_gguf')
# model.save_pretrained_gguf(gguf_save_path, tokenizer, quantization_method='q4_k_m')
# print(f"✅ GGUF modeli Google Drive'a kaydedildi: {gguf_save_path}")


### 9. Adım: 🔥 Canlı Colab Sunucusu & ngrok API Köprüsü
Bu hücreyi çalıştırdığınızda, Colab içinde OpenAI uyumlu bir API sunucusu ayağa kalkar ve ngrok üzerinden **genel bir URL** (Örn: `https://xxxx.ngrok-free.app/v1`) üretir.

Bu URL'yi kopyalayıp bilgisayarınızdaki Streamlit Chatbot arayüzünde **'🚀 Google Colab Pro / Özel Sunucu'** seçeneğini seçerek yapıştırabilir ve Colab'daki kendi modelinizi canlı olarak kullanabilirsiniz!

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import threading
import time
import os

nest_asyncio.apply()
api_app = FastAPI(title="BIST Colab LLM Server")

@api_app.get("/v1/models")
def list_models():
    return {
        "data": [
            {"id": "bist-financial-llm", "object": "model", "owned_by": "user"}
        ]
    }

@api_app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    messages = body.get("messages", [])
    temperature = float(body.get("temperature", 0.3))
    max_tokens = int(body.get("max_tokens", 700))
    rep_penalty = float(body.get("repetition_penalty", 1.18))
    
    # Mesajları modele formatla (ChatML standardı)
    prompt = ""
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    prompt += "<|im_start|>assistant\n"
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Tekrar döngülerini (repetition loop) kesinlikle engelleyen çıkarım ayarları
    outputs = model.generate(
        **inputs,
        max_new_tokens=min(max_tokens, 1200),
        temperature=max(temperature, 0.2),
        do_sample=True,
        top_p=0.9,
        repetition_penalty=rep_penalty,
        no_repeat_ngram_size=4,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    generated_full = tokenizer.batch_decode(outputs)[0]
    answer = generated_full.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    
    return {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": "bist-financial-llm",
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": answer},
                "finish_reason": "stop"
            }
        ]
    }

# Port çakışmasını engelle (önceki uvicorn süreci varsa sonlandır)
os.system("fuser -k 8000/tcp 2>/dev/null")
time.sleep(1)

# Sunucuyu arka planda çalıştır
def run_server():
    uvicorn.run(api_app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

# ngrok kimlik doğrulama tokenınızı ayarlayın
# (dashboard.ngrok.com/get-started/your-authtoken adresinden alabilirsiniz)
NGROK_TOKEN = "BURAYA_NGROK_TOKENINIZI_YAPISTIRIN"  # Kendi tokenınızı tırnak içine yapıştırın
try:
    if NGROK_TOKEN and NGROK_TOKEN != "BURAYA_NGROK_TOKENINIZI_YAPISTIRIN":
        ngrok.set_auth_token(NGROK_TOKEN)
except Exception:
    pass

# ngrok tüneli aç veya mevcut açık tüneli kullan
try:
    active_tunnels = ngrok.get_tunnels()
    if active_tunnels:
        public_url = active_tunnels[0].public_url
    else:
        public_url = ngrok.connect(8000).public_url
        
    print("\n============================================================")
    print(f"🚀 TEBRİKLER! BIST Finans LLM Sunucunuz Canlıya Alındı:")
    print(f"👉 Canlı API Adresiniz: {public_url}/v1")
    print("============================================================\n")
    print("Bu URL'yi yerel Streamlit Chatbot arayüzünüzdeki 'Colab Pro / Özel Sunucu' kutusuna yapıştırınız!")
except Exception as e:
    print(f"ngrok tüneli başlatılırken bilgi: ngrok tokenınızı ayarlamayı unutmayın: {e}")
